In [1]:
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D
using OrdinaryDiffEq
using Plots

In [78]:
#Time-saving naming scheme
const NAME_DICT = Dict{DataType, Int}()
function get_unique_name(obj)
    type = typeof(obj)
    if !haskey(NAME_DICT, type)
        NAME_DICT[type] = 0
    end
    NAME_DICT[type] +=1
    return Symbol(lowercase(string(nameof(T))), "_", NAME_DICT[type])
end
eNa = 50.0      # Sodium
eK = -80.0      # Potassium
eh = -20.0      # H-current
eleak = -50.0   # Leak current

# Conductances
gNabar = 100.0   # Sodium
gCaSbar = 3.0    # Slow calcium
gCaTbar = 1.3    # Transient calcium
gKabar = 5.0     # A-type potassium
gKCabar = 10.0   # Calcium-activated potassium
gKdrbar = 20.0   # Delayed rectifier potassium
ghbar = 0.5      # H-current
gleak = 0.01     # Leak

# Other Parameters
tauCa = 20.0     # Calcium time constant
Ca = 0.05
eCa = (500.0) * (8.6174e-5) * (283.15) * (log((3000.0 / Ca)))

134.22654278918336

In [94]:

#Base ion channel class; will be replicated for all base-class-able components
@component function BaseIonChannel(;name, v_in, conductance, reversal_potential, kwargs...)
    @parameters t
    pars = @parameters begin
        g = conductance
        E = reversal_potential
    end

    vars = @variables begin
        i(t) = 0.0
        m(t) = 0.0
        h(t) = 0.0
    end

    eqs = []

    return ODESystem(eqs, t, vars, pars; name=name)
end


BaseIonChannel (generic function with 1 method)

In [95]:
@component function CalciumDynamics(;name, v_in)
    pars = @parameters begin
        tau_Ca = 20.0  # Calcium time constant
        Ca_base = 0.05  # Base calcium level
    end
    
    vars = @variables begin
        Ca(t) = 0.05  # Calcium concentration
        E_Ca(t)       # Calcium reversal potential
    end
    
    # Get calcium currents from connected channels
    # This assumes calcium channels will connect their currents to this component
    @variables I_Ca(t)
    
    eqs = [
        # Calcium concentration dynamics
        D(Ca) ~ (1/tau_Ca) * (Ca_base + 0.94*(I_Ca) - Ca),
        
        # Dynamic calcium reversal potential using Nernst equation
        E_Ca ~ (500.0) * (8.6174e-5) * (283.15) * log((3000.0 / Ca))
    ]
    
    return ODESystem(eqs, t, vars, pars; name=name)
end

CalciumDynamics (generic function with 1 method)

In [96]:
@component function HHSodiumChannel(;name, v_in, conductance=gNabar, reversal_potential=eNa, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)

    # Activation and inactivation functions
    m_inf(v) = 1.0 / (1.0 + exp((v + 25.5) / -5.29))
    h_inf(v) = 1.0 / (1.0 + exp((v + 48.9) / 5.18))
    tau_m(v) = 1.32 - 1.26 / (1 + exp((v + 120.0) / -25.0))
    tau_h(v) = (0.67 / (1.0 + exp((v + 62.9) / -10.0))) * (1.5 + 1.0 / (1.0 + exp((v + 34.9) / 3.6)))
    
    # Channel dynamics - fixed the error in the original notebook
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^3 * base.h * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 1. Slow Calcium Channel
@component function SlowCalciumChannel(;name, v_in, conductance=gCaSbar, ca_dynamics, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=ca_dynamics.E_Ca)
    
    # Activation functions
    m_inf(v) = 1.0 / (1.0 + exp((v + 33.0) / -8.1))
    tau_m(v) = 1.4 + 7.0 / (exp((v + 27.0) / 10.0) + exp((v + 70.0) / -13.0))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        base.i ~ base.g * base.m^2 * (base.E - v_in),
        # Connect to calcium dynamics
        connect(base.i, ca_dynamics.I_Ca)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 2. Transient Calcium Channel
@component function TransientCalciumChannel(;name, v_in, conductance=gCaTbar, ca_dynamics, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=ca_dynamics.E_Ca)
    
    # Activation and inactivation functions
    m_inf(v) = 1.0 / (1.0 + exp((v + 50.0) / -7.4))
    h_inf(v) = 1.0 / (1.0 + exp((v + 78.0) / 5.0))
    tau_m(v) = 0.44 + 0.15 / (exp((v + 35.0) / 52.0) + exp((v + 35.0) / -50.0))
    tau_h(v) = 22.7 + 0.27 / (exp((v + 55.0) / 7.0) + exp((v + 55.0) / -7.0))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^3 * base.h * (base.E - v_in),
        # Connect to calcium dynamics
        connect(base.i, ca_dynamics.I_Ca)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 3. A-type Potassium Channel
@component function ATypePotassiumChannel(;name, v_in, conductance=gKabar, reversal_potential=eK, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)
    
    # Activation and inactivation functions
    m_inf(v) = 1.0 / (1.0 + exp((v + 60.0) / -8.5))
    h_inf(v) = 1.0 / (1.0 + exp((v + 78.0) / 6.0))
    tau_m(v) = 0.185 + 0.5 / (exp((v + 35.8) / 19.7) + exp((v + 79.7) / -12.7))
    tau_h(v) = 0.5 / (exp((v + 66.0) / 11.0) + exp((v + 66.0) / -11.0))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^4 * base.h * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 4. Calcium-activated Potassium Channel
@component function CalciumActivatedPotassiumChannel(;name, v_in, conductance=gKCabar, reversal_potential=eK, ca_dynamics, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)
    
    # Calcium-dependent activation
    m_inf(v, ca) = (ca / (ca + 3.0)) / (1.0 + exp((v + 28.3) / -12.6))
    tau_m(v) = 90.3 - 75.1 / (1.0 + exp((v + 46.0) / -22.7))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in, ca_dynamics.Ca) - base.m) / tau_m(v_in),
        base.i ~ base.g * base.m * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 5. Delayed Rectifier Potassium Channel
@component function DelayedRectifierPotassiumChannel(;name, v_in, conductance=gKdrbar, reversal_potential=eK, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)
    
    # Activation function
    m_inf(v) = 1.0 / (1.0 + exp((v + 12.3) / -11.8))
    tau_m(v) = 7.2 - 6.4 / (1.0 + exp((v + 28.3) / -19.2))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        base.i ~ base.g * base.m^4 * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 6. H-Current Channel
@component function HCurrentChannel(;name, v_in, conductance=ghbar, reversal_potential=eh, kwargs...)
    @parameters t
    # Base channel setup
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)
    
    # Activation function
    m_inf(v) = 1.0 / (1.0 + exp((v + 75.0) / 5.5))
    tau_m(v) = 2.0 / (exp((v + 169.7) / 11.6) + exp((v - 26.7) / -14.3))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        base.i ~ base.g * base.m * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

# 7. Leak Current
@component function LeakChannel(;name, v_in, conductance=gleak, reversal_potential=eleak, kwargs...)
    @parameters t
    # Base channel setup - leak doesn't need activation/inactivation
    @named base = BaseIonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)
    
    # Leak current is simple, just ohmic
    eqs = [
        base.i ~ base.g * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, base.states, base.ps; name=name)
end

LeakChannel (generic function with 1 method)

In [103]:
@component function HHNeuron(;name, input_current=0.0, kwargs...)
    pars = @parameters begin
        C = 0.01
        gap_C = 0.02
        #spike_threshold=-40.0
    end

    vars = @variables begin
        v(t) = -70
        i(t)#, [connect=Flow]
    end

    @named ca_dynamics = CalciumDynamics(v_in=v)

    systems = @named begin
        na = HHSodiumChannel(v_in=v)
        cas = SlowCalciumChannel(v_in=v, ca_dynamics=ca_dynamics)
        cat = TransientCalciumChannel(v_in=v, ca_dynamics=ca_dynamics)
        ka = ATypePotassiumChannel(v_in=v)
        kca = CalciumActivatedPotassiumChannel(v_in=v, ca_dynamics=ca_dynamics)
        kdr = DelayedRectifierPotassiumChannel(v_in=v)
        h = HCurrentChannel(v_in=v)
        leak = LeakChannel(v_in=v)
    end
    
    push!(systems, ca_dynamics)
    
    # Membrane potential dynamics equation
    eqs = [
        C * D(v) ~ na.i + cas.i + cat.i + ka.i + kca.i + kdr.i + h.i + leak.i + input_current
    ]
    return ODESystem(eqs, t, vars, pars; systems=systems, name=name)
end

HHNeuron (generic function with 1 method)

In [104]:
coupled = HHNeuron(name=:neur)
coupled = structural_simplify(coupled)

prob = ODEProblem(coupled,(0.0, 20))
sol = solve(prob, Tsit5())

MethodError: MethodError: no method matching ODESystem(::Vector{Any}, ::Num, ::Vector{Num}, ::Vector{Num}; name::Symbol)
The type `ODESystem` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  ODESystem(::Any, ::Any, ::Any, ::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any; ...)
   @ ModelingToolkit C:\Users\eloya\.julia\packages\ModelingToolkit\YLJ0I\src\systems\diffeqs\odesystem.jl:194
  ODESystem(::Any, ::Any, ::Any, ::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::A